# Cross-Species Consensus Peak Pipeline (v3) — Step-by-Step

**Goal:** Build a unified ATAC-seq peak set across 6 primate species with full provenance tracking.

**Pipeline steps (each callable individually):**

| Step | Function | Description |
|------|----------|-------------|
| 1 | `liftover_peaks` / `liftover_two_step` | Lift NHP peaks → hg38 |
| 2 | `summit_based_merge` + `add_peak_ids` | Merge all species into unified consensus |
| 3 | `liftback_peaks` | Lift unified peaks back to each species |
| 4 | `filter_liftback_by_size` + `reciprocal_liftover_check` | Quality-filter liftback |
| 5 | `find_species_specific_peaks` | Find NHP peaks that never reached hg38 |
| 6 | Rescue | Recover original peaks lost during liftback filtering |
| 7 | `build_master_annotation` + `classify_peak_distance` | Full annotation incl. rescued peaks & gene distances |
| 8 | Write final output | One BED per species + master annotation TSV |

**Output per species:** A single BED file containing unified + species-specific + rescued peaks.  
**Master annotation:** One TSV with all peak metadata, coordinates, gene annotation, and distance categories.

## 0. Setup & Configuration

In [ ]:
import sys, os, subprocess, tempfile, shutil
import pandas as pd
import numpy as np
from pathlib import Path

# --- Pipeline imports ---
sys.path.insert(0, os.path.abspath(".."))
from src.cross_species import (
    liftback_peaks, summit_based_merge, add_peak_ids,
    filter_liftback_by_size, reciprocal_liftover_check,
    find_species_specific_peaks, build_master_annotation,
    create_peak_annotation, extract_gene_bed_from_gtf,
    annotate_with_closest_gene, classify_peak_distance,
    resize_peaks, get_reverse_chain_file,
    REVERSE_CHAIN_FILES, DEFAULT_GTF_FILES,
)
from src.liftover import (
    liftover_peaks, liftover_two_step, get_chain_file,
    DEFAULT_CHAIN_DIR, CHAIN_FILES,
)

print("Imports OK")

In [ ]:
# =============================================================================
# PATHS — edit these to match your setup
# =============================================================================
PEAKS_BASE   = "/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
CHAIN_DIR    = "/links/groups/treutlein/USERS/jjans/data/intestine/nhp_atlas/genomes/chain_files"
LIFTOVER_PATH = "/local1/scratch/jjans/miniforge3/envs/scenicplus_scenicplus10a2_py311/bin/liftOver"

# Per-species consensus peak BEDs (native coordinates)
HUMAN_BED = f"{PEAKS_BASE}/consensus_peak_calling_Human/Consensus_Peaks_Filtered_500.bed"
SPECIES_BEDS = {
    "Bonobo":     f"{PEAKS_BASE}/consensus_peak_calling_Bonobo/Consensus_Peaks_Filtered_500.bed",
    "Chimpanzee": f"{PEAKS_BASE}/consensus_peak_calling_Chimpanzee/Consensus_Peaks_Filtered_500.bed",
    "Gorilla":    f"{PEAKS_BASE}/consensus_peak_calling_Gorilla/Consensus_Peaks_Filtered_500.bed",
    "Macaque":    f"{PEAKS_BASE}/consensus_peak_calling_Macaque/Consensus_Peaks_Filtered_500.bed",
    "Marmoset":   f"{PEAKS_BASE}/consensus_peak_calling_Marmoset/Consensus_Peaks_Filtered_500.bed",
}

# Pre-lifted BEDs (from a previous run — skip step 1 if available)
PRE_LIFTED_BEDS = {
    sp: f"{PEAKS_BASE}/cross_species_consensus_v2/01_lifted_to_human/{sp}_hg38.bed"
    for sp in SPECIES_BEDS
}

# GTF files for gene annotation
GTF_FILES = DEFAULT_GTF_FILES.copy()

# Pipeline parameters
NHP_SPECIES = list(SPECIES_BEDS.keys())
ALL_SPECIES = ["Human"] + NHP_SPECIES

MIN_MATCH = {"Bonobo": 0.9, "Chimpanzee": 0.9, "Gorilla": 0.9, "Macaque": 0.8, "Marmoset": 0.6}
SUMMIT_WINDOW      = 500     # fixed output peak width (bp)
CLUSTER_DISTANCE   = 250     # max summit-to-summit distance for merging
MAX_PEAK_SIZE      = 2000    # discard lifted peaks > this before merging
MIN_PEAK_SIZE      = 100     # discard lifted peaks < this before merging
MAX_LIFTBACK_SIZE  = 2000    # liftback filter: max
MIN_LIFTBACK_SIZE  = 100     # liftback filter: min
MAX_RECIP_DISTANCE = 500     # reciprocal check: max shift (bp)
TARGET_RESIZE      = 500     # final uniform peak width
NCPU               = 24

# Output directory — use v3 to keep v2 intact
OUTPUT_DIR = f"{PEAKS_BASE}/cross_species_consensus_v3"

# === TEST MODE: subsample to ~10k peaks per species ===
TEST_MODE  = True
TEST_NPEAKS = 10000

print(f"Output: {OUTPUT_DIR}")
print(f"Test mode: {TEST_MODE} ({TEST_NPEAKS} peaks/species)" if TEST_MODE else "")

In [ ]:
# =============================================================================
# Create output directories & optional test subset
# =============================================================================
LIFT_DIR       = os.path.join(OUTPUT_DIR, "01_lifted_to_human")
MERGED_DIR     = os.path.join(OUTPUT_DIR, "02_merged_consensus")
LIFTBACK_DIR   = os.path.join(OUTPUT_DIR, "04_lifted_back")
LIFTBACK_FILT  = os.path.join(OUTPUT_DIR, "04b_liftback_filtered")
RECIP_DIR      = os.path.join(OUTPUT_DIR, "04c_reciprocal")
SP_SPECIFIC_DIR = os.path.join(OUTPUT_DIR, "05_species_specific")
ANNOTATION_DIR = os.path.join(OUTPUT_DIR, "06_annotation")
MASTER_DIR     = os.path.join(OUTPUT_DIR, "07_master_annotation")
FINAL_DIR      = os.path.join(OUTPUT_DIR, "10_final")

for d in [LIFT_DIR, MERGED_DIR, LIFTBACK_DIR, LIFTBACK_FILT, RECIP_DIR,
          SP_SPECIFIC_DIR, ANNOTATION_DIR, MASTER_DIR, FINAL_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Subsample for testing ---
if TEST_MODE:
    SUBSET_DIR = os.path.join(OUTPUT_DIR, "00_test_subset")
    os.makedirs(SUBSET_DIR, exist_ok=True)

    # Subsample human
    sub_human = os.path.join(SUBSET_DIR, "Human_subset.bed")
    with open(HUMAN_BED) as fin, open(sub_human, "w") as fout:
        for i, line in enumerate(fin):
            if i >= TEST_NPEAKS:
                break
            fout.write(line)
    HUMAN_BED_USE = sub_human

    # Subsample NHP
    SPECIES_BEDS_USE = {}
    for sp, bed in SPECIES_BEDS.items():
        sub = os.path.join(SUBSET_DIR, f"{sp}_subset.bed")
        with open(bed) as fin, open(sub, "w") as fout:
            for i, line in enumerate(fin):
                if i >= TEST_NPEAKS:
                    break
                fout.write(line)
        SPECIES_BEDS_USE[sp] = sub
    print(f"✅ Test subsets created in {SUBSET_DIR}")
else:
    HUMAN_BED_USE = HUMAN_BED
    SPECIES_BEDS_USE = SPECIES_BEDS.copy()

# Print summary
for sp in ["Human"] + NHP_SPECIES:
    bed = HUMAN_BED_USE if sp == "Human" else SPECIES_BEDS_USE[sp]
    n = sum(1 for _ in open(bed))
    print(f"  {sp}: {n:,} peaks")

## Step 1: Liftover NHP peaks → hg38

For each non-human species, lift peaks to hg38 coordinates.  
Marmoset uses a two-step chain (calJac1 → calJac4 → hg38).

If `PRE_LIFTED_BEDS` exist from a prior run, we can re-use them (set `USE_PRE_LIFTED = True`).

In [ ]:
USE_PRE_LIFTED = not TEST_MODE  # Use pre-lifted for full run, fresh lift for test

lifted_beds = {}  # species -> path to lifted BED (hg38 coords)

for species, input_bed in SPECIES_BEDS_USE.items():
    output_bed = os.path.join(LIFT_DIR, f"{species}_hg38.bed")
    mm = MIN_MATCH[species] if isinstance(MIN_MATCH, dict) else MIN_MATCH

    # Check for pre-lifted
    if USE_PRE_LIFTED and species in PRE_LIFTED_BEDS and os.path.exists(PRE_LIFTED_BEDS[species]):
        print(f"  {species}: using pre-lifted {PRE_LIFTED_BEDS[species]}")
        shutil.copy2(PRE_LIFTED_BEDS[species], output_bed)
        lifted_beds[species] = output_bed
        continue

    print(f"\n{'='*60}")
    print(f"  Lifting {species} → hg38  (minMatch={mm})")
    print(f"{'='*60}")

    if species == "Marmoset":
        chain1 = get_chain_file("Marmoset_step1", CHAIN_DIR)
        chain2 = get_chain_file("Marmoset_step2", CHAIN_DIR)
        result = liftover_two_step(
            input_bed=input_bed, output_bed=output_bed,
            chain_file_1=chain1, chain_file_2=chain2,
            liftover_path=LIFTOVER_PATH, min_match=mm,
            auto_chr=True, verbose=True, ncpu=NCPU,
        )
    else:
        chain = get_chain_file(species, CHAIN_DIR)
        result = liftover_peaks(
            input_bed=input_bed, output_bed=output_bed,
            chain_file=chain, liftover_path=LIFTOVER_PATH,
            min_match=mm, auto_chr=True, verbose=True, ncpu=NCPU,
        )

    lifted_beds[species] = output_bed
    print(f"  → {result.get('lifted', 0):,} lifted, {result.get('unmapped', 0):,} unmapped")

print(f"\n✅ Step 1 complete: {len(lifted_beds)} species lifted")

## Step 2: Summit-based merge → unified consensus

Merge all lifted NHP peaks + Human peaks into a unified consensus set in hg38.  
Uses summit-based clustering: extract 1-bp summits → cluster within `CLUSTER_DISTANCE` → 
take score-weighted average position → expand to `SUMMIT_WINDOW` bp.

Temporary IDs (`temp_NNNNNN`) are assigned here; final IDs come in Step 6.

In [ ]:
# Collect all BED files for merging (Human + lifted NHP)
all_beds = {"Human": HUMAN_BED_USE}
all_beds.update(lifted_beds)

# Summit-based merge
merged_bed = os.path.join(MERGED_DIR, "unified_consensus_hg38_merged.bed")
merge_result = summit_based_merge(
    species_beds=all_beds,
    output_bed=merged_bed,
    summit_window=SUMMIT_WINDOW,
    cluster_distance=CLUSTER_DISTANCE,
    max_peak_size=MAX_PEAK_SIZE,
    min_peak_size=MIN_PEAK_SIZE,
    normalize_scores=True,
    verbose=True,
)
print(f"Merge result: {merge_result.get('n_consensus', 0):,} consensus peaks")

# Add temporary IDs
merged_with_ids = os.path.join(MERGED_DIR, "unified_consensus_hg38_with_ids.bed")
add_peak_ids(merged_bed, merged_with_ids, prefix="temp")

# Also create BED4 (chr, start, end, peak_id) for liftback
unified_bed4 = os.path.join(MERGED_DIR, "unified_consensus_hg38_bed4.bed")
with open(merged_with_ids) as fin, open(unified_bed4, "w") as fout:
    for line in fin:
        parts = line.strip().split("\t")
        fout.write(f"{parts[0]}\t{parts[1]}\t{parts[2]}\t{parts[3]}\n")

n_unified = sum(1 for _ in open(unified_bed4))
print(f"✅ Step 2 complete: {n_unified:,} unified consensus peaks (temp IDs)")
print(f"   Example: ", open(unified_bed4).readline().strip())

## Step 3: Liftback unified consensus → each species

For each NHP species, lift the unified hg38 consensus back to native coordinates.  
This tells us which consensus peaks have an orthologous region in each genome.

In [ ]:
liftback_results = {}

for species in NHP_SPECIES:
    output_bed = os.path.join(LIFTBACK_DIR, f"unified_consensus_{species}.bed")
    mm = MIN_MATCH[species] if isinstance(MIN_MATCH, dict) else MIN_MATCH

    result = liftback_peaks(
        input_bed=unified_bed4,
        output_bed=output_bed,
        species=species,
        chain_dir=CHAIN_DIR,
        liftover_path=LIFTOVER_PATH,
        min_match=mm,
        auto_chr=True,
        verbose=True,
        ncpu=NCPU,
    )
    liftback_results[species] = result
    n_lb = sum(1 for _ in open(output_bed)) if os.path.exists(output_bed) else 0
    print(f"  {species}: {n_lb:,} / {n_unified:,} peaks lifted back")

print(f"\n✅ Step 3 complete")

## Step 4: Quality-filter liftback peaks

**4a. Size filter:** Remove liftback peaks that became too large or too small (structural artefacts).  
**4b. Reciprocal check:** Lift species coords back to hg38, verify the round-trip lands within `MAX_RECIP_DISTANCE` of the original position.

In [ ]:
# --- 4a. Size filter ---
size_result = filter_liftback_by_size(
    liftback_dir=LIFTBACK_DIR,
    output_dir=LIFTBACK_FILT,
    species_list=NHP_SPECIES,
    min_liftback_size=MIN_LIFTBACK_SIZE,
    max_liftback_size=MAX_LIFTBACK_SIZE,
    verbose=True,
)

# --- 4b. Reciprocal liftover check ---
recip_result = reciprocal_liftover_check(
    liftback_dir=LIFTBACK_FILT,  # operate on size-filtered files
    hg38_bed=unified_bed4,
    output_dir=RECIP_DIR,
    species_list=NHP_SPECIES,
    chain_dir=CHAIN_DIR,
    liftover_path=LIFTOVER_PATH,
    min_match=MIN_MATCH,
    max_distance=MAX_RECIP_DISTANCE,
    file_pattern="unified_consensus_{species}_filtered.bed",
    verbose=True,
    ncpu=NCPU,
)

# --- Copy final filtered liftback files back to LIFTBACK_DIR for downstream ---
# (build_master_annotation reads from liftback_dir)
for species in NHP_SPECIES:
    # Prefer reciprocal pass file > size-filtered > raw liftback
    pass_file = os.path.join(RECIP_DIR, f"unified_consensus_{species}_filtered_reciprocal_pass.bed")
    filt_file = os.path.join(LIFTBACK_FILT, f"unified_consensus_{species}_filtered.bed")
    dst = os.path.join(LIFTBACK_DIR, f"unified_consensus_{species}.bed")

    if os.path.exists(pass_file) and os.path.getsize(pass_file) > 0:
        shutil.copy2(pass_file, dst)
    elif os.path.exists(filt_file):
        shutil.copy2(filt_file, dst)

    n = sum(1 for _ in open(dst)) if os.path.exists(dst) else 0
    print(f"  {species} final liftback: {n:,} peaks")

print(f"\n✅ Step 4 complete")

## Step 5: Identify species-specific peaks

For each NHP species, find original peaks that do NOT overlap any lifted-back unified peak.  
These are peaks that exist only in that species (could not be lifted to hg38 at all, or the region doesn't exist in human).

IDs: `{species}_peak_NNNNNN`

In [ ]:
species_specific_beds = {}

for species in NHP_SPECIES:
    original_bed = SPECIES_BEDS_USE[species]
    liftback_file = os.path.join(LIFTBACK_DIR, f"unified_consensus_{species}.bed")

    if not os.path.exists(liftback_file):
        print(f"  {species}: no liftback file, skipping")
        continue

    sp_specific_bed = os.path.join(SP_SPECIFIC_DIR, f"{species}_specific_peaks.bed")
    result = find_species_specific_peaks(
        original_bed=original_bed,
        liftback_bed=liftback_file,
        output_bed=sp_specific_bed,
        species=species,
        verbose=True,
    )
    species_specific_beds[species] = sp_specific_bed
    n = sum(1 for _ in open(sp_specific_bed)) if os.path.exists(sp_specific_bed) else 0
    print(f"  {species}: {n:,} species-specific peaks")

print(f"\n✅ Step 5 complete")

## Step 6: Rescue missing peaks

For each NHP species, compare the liftback peaks + species-specific peaks with the original input.
Any original peaks not covered by either set are "rescued" — peaks that lifted to hg38 successfully
but whose unified consensus region failed to lift back or was filtered out.

IDs: `{species}_rescued_NNNNNN`

In [ ]:
rescued_beds = {}  # species -> rescued BED path
rescued_rows = []  # rows to append to master annotation

for species in NHP_SPECIES:
    original_bed = SPECIES_BEDS_USE[species]
    liftback_bed = os.path.join(LIFTBACK_DIR, f"unified_consensus_{species}.bed")
    sp_specific_bed = species_specific_beds.get(species, "")

    if not os.path.exists(liftback_bed):
        continue

    with tempfile.TemporaryDirectory() as tmpdir:
        # Combine liftback + species-specific into one "covered" BED
        covered_bed = os.path.join(tmpdir, "covered.bed")
        with open(covered_bed, "w") as fout:
            for src in [liftback_bed, sp_specific_bed]:
                if src and os.path.exists(src):
                    with open(src) as fin:
                        for line in fin:
                            parts = line.strip().split("\t")
                            if len(parts) >= 3:
                                fout.write(f"{parts[0]}\t{parts[1]}\t{parts[2]}\n")

        # Extract BED3 from original, harmonize chr prefix
        orig_bed3 = os.path.join(tmpdir, "orig.bed")
        with open(original_bed) as fin, open(orig_bed3, "w") as fout:
            for line in fin:
                parts = line.strip().split("\t")
                if len(parts) >= 3:
                    fout.write(f"{parts[0]}\t{parts[1]}\t{parts[2]}\n")

        # Check chr prefix
        def _first_chr(path):
            with open(path) as f:
                for line in f:
                    if line.strip():
                        return line.split("\t")[0].startswith("chr")
            return True

        orig_has_chr = _first_chr(orig_bed3)
        cov_has_chr = _first_chr(covered_bed)
        if orig_has_chr != cov_has_chr:
            fixed = os.path.join(tmpdir, "covered_fixed.bed")
            with open(covered_bed) as fin, open(fixed, "w") as fout:
                for line in fin:
                    if orig_has_chr and not line.startswith("chr"):
                        fout.write("chr" + line)
                    elif not orig_has_chr and line.startswith("chr"):
                        fout.write(line[3:])
                    else:
                        fout.write(line)
            covered_bed = fixed

        # Sort both
        orig_sorted = os.path.join(tmpdir, "orig_sorted.bed")
        cov_sorted = os.path.join(tmpdir, "cov_sorted.bed")
        subprocess.run(f"sort -k1,1 -k2,2n {orig_bed3} > {orig_sorted}", shell=True, check=True)
        subprocess.run(f"sort -k1,1 -k2,2n {covered_bed} | bedtools merge -i - > {cov_sorted}",
                        shell=True, check=True)

        # Find originals NOT overlapping covered
        missing_bed = os.path.join(tmpdir, "missing.bed")
        subprocess.run(
            f"bedtools intersect -a {orig_sorted} -b {cov_sorted} -v > {missing_bed}",
            shell=True, check=True,
        )

        n_missing = sum(1 for _ in open(missing_bed))
        if n_missing == 0:
            print(f"  {species}: 0 peaks to rescue")
            continue

        # Resize missing peaks to TARGET_RESIZE and assign rescued IDs
        rescued_out = os.path.join(FINAL_DIR, f"{species}_rescued_peaks.bed")
        counter = 0
        sp_lower = species.lower()
        with open(missing_bed) as fin, open(rescued_out, "w") as fout:
            for line in fin:
                parts = line.strip().split("\t")
                chrom, start, end = parts[0], int(parts[1]), int(parts[2])
                center = (start + end) // 2
                new_start = max(0, center - TARGET_RESIZE // 2)
                new_end = new_start + TARGET_RESIZE
                counter += 1
                pid = f"{sp_lower}_rescued_{counter:06d}"
                fout.write(f"{chrom}\t{new_start}\t{new_end}\t{pid}\n")

                # Prepare row for master annotation
                rescued_rows.append({
                    "peak_id": pid,
                    "peak_type": f"{sp_lower}_rescued",
                    f"{species}_chr": chrom,
                    f"{species}_start": new_start,
                    f"{species}_end": new_end,
                    "species_detected": species,
                    "n_species_det": 1,
                    "n_species_orth": 0,
                    f"{species}_det": 1,
                    f"{species}_orth": 0,
                })

        rescued_beds[species] = rescued_out
        print(f"  {species}: {counter:,} rescued peaks")

print(f"\n✅ Step 6 complete: {len(rescued_rows):,} total rescued peaks")

## Step 7: Gene annotation + Master annotation

Build the **complete** master annotation table in one pass, including rescued peaks.

1. **Gene annotation**: Annotate unified and species-specific peaks with closest genes from GTFs
2. **Master annotation**: Build the full table with unified, human-specific, species-specific, **and rescued** peaks. Assigns final stable IDs (`unified_NNNNNN`, `human_peak_NNNNNN`, etc.) and tracks species detection & orthology.
3. **Distance classification**: Label each peak as promoter (<200 bp), proximal (200–2000 bp), or distal (>2000 bp)

In [ ]:
# --- 7a. Gene annotation (creates gene BEDs + annotation TSVs) ---
annotation_result = create_peak_annotation(
    unified_bed=merged_with_ids,
    human_specific_bed="",  # classified in master annotation, not separate
    species_specific_beds=species_specific_beds,
    gtf_files=GTF_FILES,
    output_dir=ANNOTATION_DIR,
    verbose=True,
)

# --- 7b. Build master annotation ---
human_annotation_tsv = os.path.join(ANNOTATION_DIR, "peak_annotation.tsv")
gene_bed_dir = os.path.join(ANNOTATION_DIR, "gene_beds")
master_output = os.path.join(MASTER_DIR, "master_annotation.tsv")

master = build_master_annotation(
    unified_bed=merged_with_ids,
    human_specific_bed="",  # human_specific is inferred from orthology
    species_specific_beds=species_specific_beds,
    liftback_dir=LIFTBACK_DIR,
    gene_bed_dir=gene_bed_dir,
    human_gene_annotation_tsv=human_annotation_tsv,
    species_list=NHP_SPECIES,
    output_file=master_output,
    verbose=True,
)

# --- 7c. Append rescued peaks ---
if rescued_rows:
    rescued_df = pd.DataFrame(rescued_rows).set_index("peak_id")
    master = pd.concat([master, rescued_df])
    print(f"  Appended {len(rescued_df):,} rescued peaks → master now {len(master):,} rows")

# --- 7d. Classify distance to nearest gene TSS ---
master = classify_peak_distance(
    master,
    promoter_threshold=200,
    proximal_threshold=2000,
)

# Show distribution
region_cols = [c for c in master.columns if c.endswith("_region")]
for rc in region_cols:
    print(f"\n  {rc}:")
    print(master[rc].value_counts().to_string())

# Save final master
master_final_path = os.path.join(MASTER_DIR, "master_annotation_final.tsv")
master.to_csv(master_final_path, sep="\t")

print(f"\n✅ Step 7 complete")
print(f"   Peak types: {master['peak_type'].value_counts().to_dict()}")
print(f"   Total peaks: {len(master):,}")
print(f"   Master saved to: {master_final_path}")
master.head(3)

## Step 8: Write final BED files

One BED per species containing **unified + species-specific + rescued** peaks,
all resized to `TARGET_RESIZE` bp.  Peak IDs match the master annotation.

**ID mapping:** The liftback files still have `temp_NNNNNN` IDs.  We load the
mapping from `build_master_annotation` to write final IDs.

In [ ]:
# Load ID mapping produced by build_master_annotation
id_mapping_file = os.path.join(MASTER_DIR, "master_annotation_id_mapping.tsv")
id_map = {}
if os.path.exists(id_mapping_file):
    mdf = pd.read_csv(id_mapping_file, sep="\t")
    id_map = dict(zip(mdf["old_id"], mdf["new_id"]))
    print(f"  Loaded ID mapping: {len(id_map):,} entries")

final_beds = {}
half = TARGET_RESIZE // 2

for species in ALL_SPECIES:
    out_bed = os.path.join(FINAL_DIR, f"all_peaks_{species}.bed")
    entries = []  # (chrom, start, end, peak_id)

    if species == "Human":
        # Unified + human-specific from master (hg38 coords)
        mask = master["peak_type"].isin(["unified", "human_specific"])
        for pid, row in master.loc[mask].iterrows():
            if pd.notna(row.get("Human_chr")):
                entries.append((
                    str(row["Human_chr"]),
                    int(row["Human_start"]),
                    int(row["Human_end"]),
                    str(pid),
                ))
    else:
        # a) Unified peaks (liftback, in species coords) — map temp IDs → final
        lb_file = os.path.join(LIFTBACK_DIR, f"unified_consensus_{species}.bed")
        if os.path.exists(lb_file):
            with open(lb_file) as f:
                for line in f:
                    parts = line.strip().split("\t")
                    if len(parts) >= 4:
                        raw_id = parts[3]
                        final_id = id_map.get(raw_id, raw_id)
                        entries.append((parts[0], int(parts[1]), int(parts[2]), final_id))

        # b) Species-specific peaks
        sp_bed = species_specific_beds.get(species, "")
        if sp_bed and os.path.exists(sp_bed):
            with open(sp_bed) as f:
                for line in f:
                    parts = line.strip().split("\t")
                    if len(parts) >= 4:
                        entries.append((parts[0], int(parts[1]), int(parts[2]), parts[3]))

        # c) Rescued peaks
        resc_bed = rescued_beds.get(species, "")
        if resc_bed and os.path.exists(resc_bed):
            with open(resc_bed) as f:
                for line in f:
                    parts = line.strip().split("\t")
                    if len(parts) >= 4:
                        entries.append((parts[0], int(parts[1]), int(parts[2]), parts[3]))

    # Resize all to uniform width
    with open(out_bed, "w") as fout:
        for chrom, start, end, pid in entries:
            center = (start + end) // 2
            ns = max(0, center - half)
            ne = ns + TARGET_RESIZE
            fout.write(f"{chrom}\t{ns}\t{ne}\t{pid}\n")

    final_beds[species] = out_bed
    print(f"  {species}: {len(entries):,} peaks → {os.path.basename(out_bed)}")

print(f"\n✅ Step 8 complete: final BEDs written to {FINAL_DIR}")

## Validation

Sanity checks:
1. **Uniform size** — every peak is exactly `TARGET_RESIZE` bp
2. **No temp IDs** — all peak IDs are final (`unified_`, `human_peak_`, `*_specific_`, `*_rescued_`)
3. **Unique IDs** — no duplicate peak IDs within a species
4. **Rescued present** — rescued peaks appear in the final BEDs
5. **Master coverage** — every peak ID in a BED also exists in the master annotation

In [ ]:
all_ok = True
master_ids = set(master.index)

for species in ALL_SPECIES:
    bed = final_beds[species]
    if not os.path.exists(bed):
        print(f"❌ {species}: BED not found")
        all_ok = False
        continue

    ids = []
    sizes = []
    temp_ids = []
    with open(bed) as f:
        for line in f:
            parts = line.strip().split("\t")
            chrom, start, end, pid = parts[0], int(parts[1]), int(parts[2]), parts[3]
            ids.append(pid)
            sizes.append(end - start)
            if pid.startswith("temp_"):
                temp_ids.append(pid)

    n = len(ids)
    unique_sizes = set(sizes)
    n_dup = n - len(set(ids))
    n_rescued = sum(1 for i in ids if "_rescued_" in i)
    not_in_master = [i for i in ids if i not in master_ids]

    # Check 1: uniform size
    if unique_sizes != {TARGET_RESIZE}:
        print(f"❌ {species}: non-uniform sizes {unique_sizes}")
        all_ok = False
    else:
        print(f"✅ {species}: {n:,} peaks, all {TARGET_RESIZE}bp", end="")

    # Check 2: no temp IDs
    if temp_ids:
        print(f"  ❌ {len(temp_ids)} temp_ IDs remain!", end="")
        all_ok = False

    # Check 3: unique IDs
    if n_dup > 0:
        print(f"  ❌ {n_dup} duplicate IDs", end="")
        all_ok = False

    # Check 4: rescued
    if species != "Human" and species in rescued_beds:
        print(f"  rescued={n_rescued}", end="")

    # Check 5: master coverage
    if not_in_master:
        print(f"  ⚠️ {len(not_in_master)} IDs not in master", end="")

    print()  # newline

if all_ok:
    print("\n🎉 All checks passed!")
else:
    print("\n⚠️ Some checks failed — review above.")